In [1]:
# Step 1a: KNN imputation of missing DNAm values (k = 15)
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

K_NEIGHBORS = 15            # Methods: "k-nearest neighbours (k = 15)"
N_RAW_EXPECTED = 2547       # rows in the raw TCGA file
N_COHORT_EXPECTED = 2546    # cohort size stated in Methods


def load_methylation_data(file_path):
    """Raw file: one row per sample, one column per CpG, no header, no IDs."""
    try:
        data = pd.read_csv(file_path, header=None)
        print(f"Successfully loaded data with shape: {data.shape}")
        return data
    except Exception as e:
        print(f"Error loading data: {str(e)}")
        return None


def exclude_last_sample(data):
    """Explicit, documented exclusion of the last raw row.
    The last raw row is TCGA-NC-A5HT (last row of TCGA_clinic2.csv, whose row
    order matches this matrix). It is excluded, leaving the 2,546 individuals
    used in all downstream analyses and stated in Methods. The first row
    (TCGA-DE-A4MB) is kept."""
    assert data.shape[0] == N_RAW_EXPECTED, (
        f"Expected {N_RAW_EXPECTED} raw rows, found {data.shape[0]}")
    data = data.iloc[:-1].reset_index(drop=True)
    assert data.shape[0] == N_COHORT_EXPECTED
    print(f"Excluded last sample (TCGA-NC-A5HT); cohort size = {data.shape[0]}")
    return data


def impute_methylation_data(data, k_neighbors=K_NEIGHBORS):
    """Impute missing values with KNN; observed values are left unchanged."""
    total_values = data.size
    missing_values = int(data.isna().sum().sum())

    imputer = KNNImputer(n_neighbors=k_neighbors)
    imputed_data = pd.DataFrame(
        imputer.fit_transform(data), index=data.index, columns=data.columns
    )

    info = {
        'total_values': total_values,
        'missing_values_before': missing_values,
        'missing_percentage_before': 100 * missing_values / total_values,
        'k_neighbors': k_neighbors,
        'missing_values_after': int(imputed_data.isna().sum().sum()),
    }
    return imputed_data, info


from google.colab import files


def main():
    print("Please upload your methylation CSV file (TCGA_DNAm_processed.csv)...")
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]

    data = load_methylation_data(file_path)
    if data is None:
        return

    data = exclude_last_sample(data)

    print("\nPerforming imputation...")
    imputed_data, info = impute_methylation_data(data)

    if info['missing_values_before'] == 0:
        print("NOTE: the input has no missing values, so imputation changed nothing.")

    # Saved WITHOUT header and index. Downstream mapping.ipynb must therefore
    # read this file with header=None.
    output_path = "DNAm_processed_imputed.csv"
    imputed_data.to_csv(output_path, index=False, header=False)
    files.download(output_path)

    print(f"\nAnalysis complete! Results saved and ready to download: {output_path}")
    print("\nSummary:")
    print(f"Total samples: {len(data)}")
    print(f"Total CpG sites: {len(data.columns)}")
    print(f"k neighbours: {info['k_neighbors']}")
    print(f"Missing values before imputation: {info['missing_values_before']}")
    print(f"Missing values after imputation: {info['missing_values_after']}")


if __name__ == "__main__":
    main()

Please upload your methylation CSV file (TCGA_DNAm_processed.csv)...


Saving TCGA_DNAm_processed.csv to TCGA_DNAm_processed.csv
Successfully loaded data with shape: (2547, 2000)
Excluded last sample (TCGA-NC-A5HT); cohort size = 2546

Performing imputation...
NOTE: the input has no missing values, so imputation changed nothing.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Analysis complete! Results saved and ready to download: DNAm_processed_imputed.csv

Summary:
Total samples: 2546
Total CpG sites: 2000
k neighbours: 15
Missing values before imputation: 0
Missing values after imputation: 0
